# 13 · Capstone: a board assistant for Halvora

**Goal:** combine what the earlier notebooks taught into one small, working application. It answers a board member's questions about the Halvora annual report, cites every claim, does its arithmetic in code, checks its own answers, and is measured by an eval.

| | |
|---|---|
| **You will build** | A router, a search tool that returns citable results, a calculator tool, an agent loop, a reviewer, and an eval suite |
| **Prerequisites** | Notebooks 02, 03, 05 and 08 especially. The rest helps |
| **Run time** | About 3 minutes |
| **Cost** | Under $0.20 expected. This notebook stops itself at $0.40 |
| **Paid cells** | Sections 4 to 7. Sections 1 to 3 are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## What we're building

```
 question ──▶ 1. Router            Is this about the report? (structured output, notebook 02)
                  │ no ──▶ polite refusal, no further cost
                  ▼ yes
              2. Agent loop        Claude decides which tools to call (notebook 05)
                  ├── search_report   finds sections with keyword search (notebook 08)
                  │                   and returns them as citable search results
                  └── calculate       does arithmetic in code, not in Claude's head
                  ▼
              3. Cited answer      every claim points to a report section (notebook 09)
                  ▼
              4. Reviewer          checks the answer against its sources, sends one retry
                  ▼                if needed (evaluator-optimizer, notebook 11)
              5. Eval              scores the whole system on fixed questions (notebook 03)
```

Most of this is a **workflow**: the router, the review and the retry are fixed steps in our code. Only step 2 is an **agent**, because the questions vary too much to script which tools to call. That's the advice from notebook 11: use the simplest design that works, and add an agent only where it's needed.

## Setup

The same setup as the earlier notebooks, plus the conversation helpers from notebook 01. Run the cells from top to bottom.

In [1]:
import ast
import json
import math
import operator
import os
import re
from collections import Counter
from pathlib import Path
from typing import Literal

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [3]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.40     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"   [input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    messages.append({"role": "assistant", "content": response.content})

## 1. The knowledge base (free)

We split the report into its `## ` sections, as in notebook 07. Each section keeps its title, because titles are strong clues for search, and makes a good label for a citation.

In [4]:
report = Path("data/halvora_report.md").read_text(encoding="utf-8")

sections = []
for chunk in re.split(r"\n(?=## )", report)[1:]:   # [1:] skips the title block
    title = chunk.splitlines()[0].lstrip("# ").strip()
    body = chunk.split("\n", 1)[1].strip()
    sections.append({"title": title, "body": body})

for number, section in enumerate(sections):
    print(f"{number:>2}. {section['title']:<38} {len(section['body']):>5,} characters")

assert len(sections) >= 12

 0. Executive Summary                        829 characters
 1. Table of Contents                        407 characters
 2. Methodology                              825 characters
 3. Section 1: Medical Research            1,046 characters
 4. Section 2: Software Engineering        1,187 characters
 5. Section 3: Financial Analysis          1,064 characters
 6. Section 4: Product Development           952 characters
 7. Section 5: Legal Developments            954 characters
 8. Section 6: Human Resources               759 characters
 9. Section 7: Marketing and Brand           710 characters
10. Section 8: Supply Chain and Operations   701 characters
11. Section 9: Sustainability                656 characters
12. Section 10: Cybersecurity Analysis     1,132 characters
13. Section 11: Research Partnerships        551 characters
14. Section 12: Future Outlook               525 characters


## 2. Search: BM25, from notebook 08 (free)

For search we use **BM25**, the keyword search from notebook 08. It's free, instant and good with exact IDs such as `INC-2023-Q4-011`. Notebook 08 showed that adding embeddings (hybrid search) also catches paraphrases. We leave that out here, so this notebook needs no Voyage key, and you can try adding it in Your turn.

In [5]:
STOPWORDS = set("""a an and are as at be by did do does for from had has have how i in is it its
many much of on or our over so than that the their there these they this to was we were what
when where which who why will with you your""".split())


def tokenize(text):
    # Keep IDs such as "INC-2023-Q4-011" and "XDR-47" as one token.
    words = re.findall(r"[a-z0-9]+(?:[-.][a-z0-9]+)*", text.lower())
    return [w for w in words if w not in STOPWORDS]


class BM25Index:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.documents = documents
        self.doc_tokens = [tokenize(doc) for doc in documents]
        self.doc_freq = Counter(term for tokens in self.doc_tokens for term in set(tokens))
        self.avg_len = sum(len(tokens) for tokens in self.doc_tokens) / len(self.doc_tokens)
        self.k1, self.b = k1, b

    def idf(self, term):
        n, N = self.doc_freq[term], len(self.documents)
        return math.log((N - n + 0.5) / (n + 0.5) + 1)

    def score(self, query, i):
        tf = Counter(self.doc_tokens[i])
        length_factor = 1 - self.b + self.b * len(self.doc_tokens[i]) / self.avg_len
        return sum(self.idf(t) * tf[t] * (self.k1 + 1) / (tf[t] + self.k1 * length_factor)
                   for t in set(tokenize(query)) if t in tf)

    def search(self, query, k=3):
        """Return the positions of the k best-matching documents, best first. Zero scores are dropped."""
        scored = [(self.score(query, i), i) for i in range(len(self.documents))]
        scored = [pair for pair in scored if pair[0] > 0]
        scored.sort(reverse=True)
        return [i for _, i in scored[:k]]


bm25 = BM25Index([f"{s['title']}\n{s['body']}" for s in sections])

for query in ["INC-2023-Q4-011", "revenue operating margin", "people hired new office"]:
    print(f"{query!r:<30} -> {[sections[i]['title'] for i in bm25.search(query)]}")

'INC-2023-Q4-011'              -> ['Section 10: Cybersecurity Analysis', 'Section 12: Future Outlook', 'Section 2: Software Engineering']
'revenue operating margin'     -> ['Executive Summary', 'Section 3: Financial Analysis', 'Section 12: Future Outlook']
'people hired new office'      -> ['Section 6: Human Resources', 'Executive Summary', 'Methodology']


## 3. The tools (free)

### Tool 1: `search_report`, returning citable search results

Notebook 09 used citations on whole documents. Here the documents come from a **tool**, so we return them as `search_result` blocks inside the `tool_result`. Claude can then cite them, and each citation carries the `source` and `title` we set.

- Each section's paragraphs become separate text blocks. Claude cites whole blocks, so **smaller blocks give more precise citations**.
- `"citations": {"enabled": True}` is needed on every search result.
- If nothing matches, we return a plain text block saying so, not an error.

### Tool 2: `calculate`, arithmetic done by code

Notebook 04's meal plans kept getting their own numbers wrong, so here arithmetic goes to code. **Never pass model-written text to `eval()`**: it would run any Python, including harmful code. Our calculator reads the expression with `ast` and allows only numbers and `+ - * / ( )`.

In [6]:
def search_report(query):
    """Find the best-matching report sections, as citable search_result blocks."""
    hits = bm25.search(query, k=3)
    if not hits:
        return [{"type": "text", "text": "No matching sections found."}]
    results = []
    for i in hits:
        paragraphs = [p.strip() for p in sections[i]["body"].split("\n\n") if p.strip()]
        results.append({
            "type": "search_result",
            "source": f"halvora-report#section-{i}",
            "title": sections[i]["title"],
            "content": [{"type": "text", "text": paragraph} for paragraph in paragraphs],
            "citations": {"enabled": True},
        })
    return results


ALLOWED_OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                     ast.Div: operator.truediv, ast.USub: operator.neg}


def calculate(expression):
    """Safely evaluate arithmetic such as '(11.4 - 9.1) / 9.1 * 100'. Nothing else is allowed."""
    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.operand))
        raise ValueError("Only numbers and + - * / ( ) are allowed.")
    return round(evaluate(ast.parse(expression, mode="eval").body), 4)


tools = [
    {
        "name": "search_report",
        "description": "Search Halvora's 2023 annual report. Use this before answering any question about Halvora, "
                       "and search again with different words if the first results don't contain the answer.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Keywords to search for, e.g. 'revenue growth'"}},
            "required": ["query"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression. Use this for every calculation, "
                       "such as percentage changes, instead of working it out yourself.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Numbers and + - * / ( ) only"}},
            "required": ["expression"],
        },
    },
]


def run_tool(name, tool_input):
    if name == "search_report":
        return search_report(**tool_input)          # already a list of content blocks
    if name == "calculate":
        return [{"type": "text", "text": str(calculate(**tool_input))}]
    raise ValueError(f"Unknown tool: {name}")


# Free local checks.
print(search_report("INC-2023-Q4-011")[0]["title"])
print("Margin change in percentage points:", calculate("11.4 - 9.1"))
print("Relative change in the margin, %   :", calculate("(11.4 - 9.1) / 9.1 * 100"))
for bad in ["__import__('os').system('ls')", "2 ** 1000000"]:
    try:
        calculate(bad)
        raise AssertionError("This should have been refused")
    except ValueError as error:
        print(f"Refused {bad!r}: {error}")

Section 10: Cybersecurity Analysis
Margin change in percentage points: 2.3
Relative change in the margin, %   : 25.2747
Refused "__import__('os').system('ls')": Only numbers and + - * / ( ) are allowed.
Refused '2 ** 1000000': Only numbers and + - * / ( ) are allowed.


## 4. The router (paid)

The first step is a cheap gate. A structured output with a `Literal` forces one of two labels (notebook 02). Questions that aren't about Halvora get a polite answer straight away, without running searches or the agent. This saves money and keeps the assistant on topic.

In [7]:
class Route(BaseModel):
    category: Literal["about_halvora", "unrelated"]


def route(question):
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=500,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": (
            "You route questions for an assistant that answers questions about Halvora Systems' 2023 annual report. "
            "Label the question about_halvora if it asks about Halvora, its divisions, finances, people, research, "
            "security or plans. Label it unrelated otherwise.\n\n"
            f"<question>{question}</question>"
        )}],
        output_format=Route,
    ))
    return response.parsed_output.category


for question in ["How much did revenue grow in 2023?", "Write me a poem about cats."]:
    print(f"{question!r} -> {route(question)}")

   [input=338 output=14 cost=$0.00082 | spent $0.0008 of $0.40]
'How much did revenue grow in 2023?' -> about_halvora
   [input=335 output=13 cost=$0.00080 | spent $0.0016 of $0.40]
'Write me a poem about cats.' -> unrelated


## 5. The agent loop and cited answers (paid)

This is the tool loop from notebook 05, with a system prompt that sets the rules. The answer arrives as several text blocks, and the ones based on our search results carry `search_result_location` citations.

`answer_question` returns the final response, and `render` prints it with `[n]` markers and a source list, like the renderer in notebook 09.

In [8]:
SYSTEM = """You answer questions from Halvora Systems' board about the company's 2023 annual report.
- Use search_report to find information. Answer only from what it returns, and cite it.
- Use calculate for every calculation. Never do arithmetic in your head.
- If the report doesn't contain the answer, say so plainly and don't guess.
- Keep answers to at most three sentences."""


def answer_question(question, extra_instruction=None, max_turns=6):
    messages = []
    add_user_message(messages, question if not extra_instruction else f"{question}\n\n{extra_instruction}")
    for turn in range(1, max_turns + 1):
        response = track(client.messages.create(
            model=MODEL,
            max_tokens=1500,
            output_config={"effort": "low"},
            system=SYSTEM,
            messages=messages,
            tools=tools,
        ))
        add_assistant_message(messages, response)
        if response.stop_reason != "tool_use":
            return response
        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"   tool: {block.name}({block.input})")
                try:
                    content = run_tool(block.name, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": content})
                except Exception as error:
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": f"Error: {error}", "is_error": True})
        add_user_message(messages, results)
    raise RuntimeError(f"No final answer after {max_turns} turns.")


def full_text(response):
    """Join every text block. A cited answer is split across many small blocks."""
    return "".join(block.text for block in response.content if block.type == "text")


def citations_of(response):
    return [c for block in response.content if block.type == "text" for c in (block.citations or [])]


def render(response):
    answer, sources = "", []
    for block in response.content:
        if block.type != "text":
            continue
        answer += block.text
        for citation in block.citations or []:
            sources.append(citation)
            answer += f" [{len(sources)}]"
    print("\nAnswer:", answer.strip())
    for n, c in enumerate(sources, 1):
        print(f"   [{n}] {c.title}: \"{' '.join(c.cited_text.split())[:120]}\"")


response = answer_question("By how many percentage points did Halvora's operating margin improve in 2023?")
render(response)

assert response.stop_reason == "end_turn"
assert citations_of(response), "Expected at least one citation"

   [input=694 output=106 cost=$0.00245 | spent $0.0041 of $0.40]
   tool: search_report({'query': 'operating margin'})
   tool: search_report({'query': 'operating income and revenue 2023 2022'})
   [input=3,804 output=56 cost=$0.00817 | spent $0.0122 of $0.40]
   tool: calculate({'expression': '11.4 - 9.1'})
   [input=3,868 output=87 cost=$0.00861 | spent $0.0208 of $0.40]

Answer: Halvora's operating margin improved by 2.3 percentage points in 2023. It rose from 9.1% to 11.4%, mainly because of lower cloud costs and fewer customer-facing outages. [1]
   [1] Executive Summary: "2023 was a year of stabilisation and focused growth. Revenue grew 14% to $412 million. Operating margin improved from 9."


**What to notice**

- Claude chose the tools: probably a search, then `calculate` for the difference. Nothing in our code told it which to call. That's the agent part.
- Each `[n]` points to a paragraph that really is in the report, because citations quote our own search results.

## 6. The reviewer: evaluator-optimizer (paid)

A second, separate call checks the answer against the exact passages it cites. If a claim isn't supported, the reviewer's feedback goes back to the agent for **one** retry. The limit matters: without it, a strict reviewer and a stubborn agent could loop and keep spending.

In [9]:
class Review(BaseModel):
    supported: bool
    problems: list[str]


def review(question, response):
    cited = "\n".join(f"- {c.title}: {c.cited_text}" for c in citations_of(response))
    result = track(client.messages.parse(
        model=MODEL,
        max_tokens=800,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": f"""Check this answer for a board member.

<question>{question}</question>
<answer>{full_text(response)}</answer>
<cited_passages>
{cited or "(none)"}
</cited_passages>

Is every factual claim and number in the answer supported by the cited passages,
or correctly calculated from them? Saying the report doesn't cover something is fine.
List any problems."""}],
        output_format=Review,
    ))
    return result.parsed_output


def board_assistant(question):
    """The whole pipeline: route, answer, review, and retry once if needed."""
    print(f"\nQ: {question}")
    if route(question) == "unrelated":
        print("   routed: unrelated, so no search was run")
        return {"answer": "I can only answer questions about Halvora's 2023 annual report.",
                "response": None, "routed": "unrelated", "retried": False}

    response = answer_question(question)
    verdict = review(question, response)
    retried = False
    if not verdict.supported:
        print(f"   reviewer found problems: {verdict.problems}")
        feedback = "A reviewer found these problems in a previous answer: " + "; ".join(verdict.problems)
        response = answer_question(question, extra_instruction=feedback)
        retried = True
    render(response)
    return {"answer": full_text(response), "response": response, "routed": "about_halvora", "retried": retried}


result = board_assistant("What happened in incident INC-2023-Q4-011, and how quickly was it contained?")
assert result["routed"] == "about_halvora"


Q: What happened in incident INC-2023-Q4-011, and how quickly was it contained?
   [input=353 output=14 cost=$0.00085 | spent $0.0217 of $0.40]
   [input=698 output=57 cost=$0.00197 | spent $0.0237 of $0.40]
   tool: search_report({'query': 'INC-2023-Q4-011'})
   [input=2,886 output=241 cost=$0.00818 | spent $0.0318 of $0.40]
   [input=931 output=16 cost=$0.00202 | spent $0.0339 of $0.40]

Answer: On 14 November 2023, security monitoring detected unusual login activity on a legacy reporting service, and the incident was classified as severity 1. [1] An attacker exploited a known weakness in an unpatched authentication library and had access to internal reporting dashboards for about 31 hours. Forensic experts found no evidence that customer data was copied out, but they couldn't fully rule it out, so affected customers and regulators were notified. [2] The incident was contained within 36 hours. The response team rotated all service credentials, patched the library across 140 services

## 7. Evaluating the whole system (paid)

As notebook 03 taught, an app isn't finished until it's measured. Each test case says what a good answer must contain, checked by code:

- `must_include`: strings the answer must contain, such as a figure from the report,
- `route`: the label the router should give,
- `expect_citations`: whether the answer should cite the report. A question the report can't answer should get **no** citations, and no invented facts.

In [ ]:
test_cases = [
    {"question": "How much did revenue grow in 2023, and what was the total?",
     "must_include": ["14", "412"], "route": "about_halvora", "expect_citations": True},
    {"question": "How many people did Halvora hire, and where did it open an office?",
     "must_include": ["310", "Pune"], "route": "about_halvora", "expect_citations": True},
    {"question": "What is XDR-47?",
     "must_include": ["XDR-47"], "route": "about_halvora", "expect_citations": True},
    {"question": "What was the CEO's salary in 2023?",
     "must_include": [], "route": "about_halvora", "expect_citations": False},
    {"question": "What's a good recipe for banana bread?",
     "must_include": [], "route": "unrelated", "expect_citations": False},
]

eval_results = []
for case in test_cases:
    spent_before = spent
    result = board_assistant(case["question"])
    n_citations = len(citations_of(result["response"])) if result["response"] else 0
    checks = {
        "route": result["routed"] == case["route"],
        "content": all(s.lower() in result["answer"].lower() for s in case["must_include"]),
        "citations": (n_citations > 0) == case["expect_citations"],
    }
    eval_results.append({"case": case, "checks": checks, "cost": spent - spent_before, "retried": result["retried"]})

print(f"\n{'route':>6}{'content':>9}{'cites':>7}{'retry':>7}{'cost':>9}  question")
for r in eval_results:
    c = r["checks"]
    marks = [("ok" if c[k] else "FAIL") for k in ("route", "content", "citations")]
    print(f"{marks[0]:>6}{marks[1]:>9}{marks[2]:>7}{('yes' if r['retried'] else 'no'):>7}{r['cost']:>9.4f}  {r['case']['question'][:55]}")

passed = sum(all(r["checks"].values()) for r in eval_results)
print(f"\nPassed {passed}/{len(eval_results)} test cases. This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

assert len(eval_results) == len(test_cases)

**What to notice**

- **Look at the `cost` column.** The unrelated question cost almost nothing, because the router stopped it early. The other questions cost the most where the agent searched more than once.
- **The CEO's salary isn't in the report.** A good answer says so, with no citations and no invented number. That case checks the most important safety property of a RAG assistant.
- **A `FAIL` is information, not a disaster.** Read the answer, decide whether the system or the test was wrong, fix one of them, and run the eval again. That's the loop from notebooks 03 and 04.

## What a production version would add

| Gap | Where you learned the fix |
|---|---|
| Keyword search misses paraphrases | Hybrid search with embeddings, and reranking (notebooks 07 and 08) |
| The same system prompt and tools are resent every call | Prompt caching (notebook 09) |
| Other apps should be able to use `search_report` | Expose it from an MCP server (notebook 10) |
| Answers take a few seconds | Streaming (notebook 01) |
| Five test cases is a small exam | A bigger eval set, including real board questions (notebook 03) |

## Takeaways

1. **Real apps combine small techniques.** Each part here came from one earlier notebook.
2. **Use a workflow wherever the steps are known**, and an agent only where they aren't.
3. **`search_result` blocks** let Claude cite content that your own tools return.
4. **Send arithmetic to code, and never `eval()` model output.**
5. **A reviewer with a retry limit** catches mistakes without risking a costly loop.
6. **Measure the whole system** with an eval, not just individual prompts.

## Your turn

Each exercise costs a few cents.

- Add a sixth test case of your own, ideally one you expect to fail, and see whether it does.
- Swap BM25 for the hybrid retriever from notebook 08, then re-run the eval. Does anything change?
- Add `"cache_control": {"type": "ephemeral"}` to the last tool definition, then compare `input` on repeated questions. Check `cache_read_input_tokens` to see whether the prefix was long enough to cache.
- Make the reviewer stricter, for example by requiring a citation for every sentence. How often does it trigger a retry now, and what does that cost?

**This is the last notebook.** Thanks for working through the series. The [README](../README.md) lists every notebook, if you want to revisit one.